# 05 -- SciPy

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- utiliser `scipy.optimize` pour trouver des racines et minimiser des fonctions ;
- resoudre des systemes d'equations lineaires et non-lineaires ;
- effectuer des interpolations avec `scipy.interpolate` ;
- calculer des integrales numeriques avec `scipy.integrate` ;
- appliquer des filtres et des transformees de Fourier avec `scipy.signal` et `scipy.fft` ;
- realiser des tests statistiques avec `scipy.stats` ;
- travailler avec des matrices creuses (`scipy.sparse`).

## Prerequis -- ce que vous devez maitriser

Ce notebook fait partie des **Side tracks**. Vous devez etre a l'aise avec :

- les types de base Python et les fonctions typees ;
- NumPy : `ndarray`, broadcasting, vectorisation (voir `01_NumPy_approfondi.ipynb`) ;
- les bases de l'algebre lineaire (vecteurs, matrices, systemes) ;

Ce que ce notebook **ne couvrira pas** :

- le machine learning (voir `07_Scikit_learn_intro.ipynb`) ;
- la visualisation avancee (voir `Data_visualization/`) ;
- Pandas ou Polars (voir notebooks dedies).

## Plan

1. Installation et import
2. `scipy.optimize` -- minimisation et racines
3. `scipy.optimize` -- moindres carres (curve fitting)
4. `scipy.interpolate` -- interpolation
5. `scipy.integrate` -- integration numerique
6. `scipy.integrate` -- EDO (equations differentielles)
7. `scipy.linalg` -- algebre lineaire avancee
8. `scipy.stats` -- distributions et tests
9. `scipy.signal` -- filtrage
10. `scipy.fft` -- transformee de Fourier
11. `scipy.sparse` -- matrices creuses
12. `scipy.spatial` -- distances et arbres
13. Synthese
14. Exercices

---

## 1. Installation et import

SciPy s'installe avec `pip install scipy` (ou `uv add scipy`). L'import se fait module par module :

In [ ]:
import numpy as np
import scipy
scipy.__version__

On n'importe **jamais** `from scipy import *`. On importe le sous-module dont on a besoin :

In [ ]:
from scipy import optimize, interpolate, integrate, stats, linalg, signal, fft, sparse

---

## 2. `scipy.optimize` -- minimisation et racines

### Trouver une racine avec `brentq`

In [ ]:
from scipy.optimize import brentq

def f(x):
    return x**3 - 2*x - 5

racine = brentq(f, 2, 3)
racine

`brentq` combine bisection et interpolation. Il faut fournir un intervalle `[a, b]` ou `f(a)` et `f(b)` sont de signes opposes.

### Trouver une racine avec `fsolve` (Newton multidimensionnel)

In [ ]:
from scipy.optimize import fsolve

def equations(vars):
    x, y = vars
    return [x + y - 3, x**2 + y**2 - 5]

solution = fsolve(equations, [1, 1])
solution

### Minimiser une fonction scalaire

In [ ]:
from scipy.optimize import minimize_scalar

def g(x):
    return (x - 3)**2 + 1

result = minimize_scalar(g)
result.x, result.fun

### Minimiser une fonction multivariee

In [ ]:
from scipy.optimize import minimize

def rosenbrock(xy):
    x, y = xy
    return (1 - x)**2 + 100*(y - x**2)**2

result = minimize(rosenbrock, x0=[0, 0], method='Nelder-Mead')
result.x

La fonction de Rosenbrock est un classique du benchmarking d'optimiseurs. Le minimum global est en `(1, 1)`.

---

## 3. `scipy.optimize` -- moindres carres (curve fitting)

`curve_fit` ajuste une fonction parametrique aux donnees.

In [ ]:
from scipy.optimize import curve_fit

def modele(x, a, b, c):
    return a * np.exp(-b * x) + c

np.random.seed(42)
x_data = np.linspace(0, 4, 50)
y_data = modele(x_data, 2.5, 1.3, 0.5) + 0.2 * np.random.normal(size=50)

params, covariance = curve_fit(modele, x_data, y_data)
print(f'a={params[0]:.3f}, b={params[1]:.3f}, c={params[2]:.3f}')

La matrice de covariance `covariance` permet de calculer l'incertitude sur les parametres.

---

## 4. `scipy.interpolate` -- interpolation

### Interpolation 1D

In [ ]:
from scipy.interpolate import interp1d

x = np.array([0, 1, 2, 3, 4, 5])
y = np.array([0, 0.8, 0.9, 0.1, -0.8, -1.0])

f_linear = interp1d(x, y, kind='linear')
f_cubic = interp1d(x, y, kind='cubic')

x_new = np.linspace(0, 5, 100)
f_linear(2.5), f_cubic(2.5)

### Splines cubiques

In [ ]:
from scipy.interpolate import CubicSpline

cs = CubicSpline(x, y)
cs(2.5)

`CubicSpline` est preferable a `interp1d(kind='cubic')` pour les nouvelles applications.

---

## 5. `scipy.integrate` -- integration numerique

### Integrale definie avec `quad`

In [ ]:
from scipy.integrate import quad

def f(x):
    return np.exp(-x**2)

resultat, erreur = quad(f, 0, np.inf)
resultat

`quad` retourne un tuple `(valeur, estimation_erreur)`. Il gere les bornes infinies.

### Integration sur un tableau de valeurs avec `trapezoid`

In [ ]:
from scipy.integrate import trapezoid

x = np.linspace(0, np.pi, 100)
y = np.sin(x)
trapezoid(y, x)

Pour un tableau de points (pas une fonction), `trapezoid` (anciennement `trapz`) utilise la regle des trapezes.

---

## 6. `scipy.integrate` -- equations differentielles (EDO)

`solve_ivp` resout des problemes a valeur initiale.

In [ ]:
from scipy.integrate import solve_ivp

# dy/dt = -2y, y(0) = 1  ->  solution analytique : y = exp(-2t)
def ode(t, y):
    return -2 * y

sol = solve_ivp(ode, t_span=[0, 5], y0=[1.0], dense_output=True)
t = np.linspace(0, 5, 100)
sol.sol(t)[0][:5]  # premieres valeurs

Le parametre `dense_output=True` permet d'evaluer la solution en n'importe quel point.

### Systeme d'EDO : oscillateur harmonique

In [ ]:
def oscillateur(t, state):
    x, v = state
    dxdt = v
    dvdt = -x  # k/m = 1
    return [dxdt, dvdt]

sol = solve_ivp(oscillateur, [0, 20], [1.0, 0.0], max_step=0.1)
sol.y[0][:5]  # position

---

## 7. `scipy.linalg` -- algebre lineaire avancee

`scipy.linalg` etend `numpy.linalg` avec des decompositions supplementaires.

In [ ]:
from scipy.linalg import lu, svd, cholesky

A = np.array([[2.0, 1.0], [1.0, 3.0]])

# Decomposition LU
P, L, U = lu(A)
print('L =', L)
print('U =', U)

In [ ]:
# Decomposition SVD
U, s, Vt = svd(A)
print('Valeurs singulieres :', s)

In [ ]:
# Decomposition de Cholesky (matrice symetrique definie positive)
L = cholesky(A, lower=True)
L

### Resoudre un systeme bande

In [ ]:
from scipy.linalg import solve_banded

# Matrice tridiagonale : on la stocke en format bande
ab = np.array([[0, -1, -1, -1],   # diagonale superieure
               [2,  2,  2,  2],   # diagonale principale
               [-1, -1, -1,  0]]) # diagonale inferieure
b = np.array([1.0, 2.0, 3.0, 4.0])
solve_banded((1, 1), ab, b)

---

## 8. `scipy.stats` -- distributions et tests statistiques

### Distributions

In [ ]:
from scipy.stats import norm

# Distribution normale standard
norm.pdf(0)      # densite en 0

In [ ]:
norm.cdf(1.96)   # probabilite P(X <= 1.96)

In [ ]:
norm.ppf(0.975)  # quantile : quelle valeur donne P(X <= x) = 0.975 ?

In [ ]:
norm.rvs(size=5, random_state=42)  # 5 tirages aleatoires

### Test de Student (t-test)

In [ ]:
from scipy.stats import ttest_ind

rng = np.random.default_rng(42)
groupe_a = rng.normal(loc=5.0, scale=1.0, size=30)
groupe_b = rng.normal(loc=5.5, scale=1.0, size=30)

stat, p_value = ttest_ind(groupe_a, groupe_b)
print(f't-stat = {stat:.3f}, p-value = {p_value:.4f}')

Si `p_value < 0.05`, on rejette l'hypothese nulle (les moyennes sont significativement differentes).

### Test de Shapiro-Wilk (normalite)

In [ ]:
from scipy.stats import shapiro

stat, p_value = shapiro(groupe_a)
print(f'stat = {stat:.4f}, p-value = {p_value:.4f}')

### Test du chi-deux

In [ ]:
from scipy.stats import chi2_contingency

observe = np.array([[10, 20, 30], [6, 9, 17]])
chi2, p, dof, expected = chi2_contingency(observe)
print(f'chi2 = {chi2:.3f}, p = {p:.4f}, dof = {dof}')

---

## 9. `scipy.signal` -- filtrage

### Filtre passe-bas de Butterworth

In [ ]:
from scipy.signal import butter, sosfilt

# Signal bruite
rng = np.random.default_rng(42)
t = np.linspace(0, 1, 1000)
signal_pur = np.sin(2 * np.pi * 5 * t)  # 5 Hz
bruit = 0.5 * rng.normal(size=1000)
signal_bruite = signal_pur + bruit

# Filtre passe-bas a 10 Hz (frequence d'echantillonnage = 1000 Hz)
sos = butter(4, 10, btype='low', fs=1000, output='sos')
signal_filtre = sosfilt(sos, signal_bruite)
signal_filtre[:5]

### Detection de pics

In [ ]:
from scipy.signal import find_peaks

x = np.sin(np.linspace(0, 4 * np.pi, 200))
peaks, properties = find_peaks(x, height=0.5)
peaks

---

## 10. `scipy.fft` -- transformee de Fourier

In [ ]:
from scipy.fft import fft, fftfreq

N = 600
T = 1.0 / 800.0  # pas temporel
t = np.arange(N) * T
signal_mix = np.sin(50.0 * 2.0 * np.pi * t) + 0.5 * np.sin(80.0 * 2.0 * np.pi * t)

yf = fft(signal_mix)
xf = fftfreq(N, T)[:N//2]

# Les deux pics sont a 50 Hz et 80 Hz
amplitudes = 2.0 / N * np.abs(yf[:N//2])
amplitudes[:5]

La FFT decompose un signal temporel en ses composantes frequentielles.

---

## 11. `scipy.sparse` -- matrices creuses

Une matrice **creuse** a la majorite de ses elements egaux a zero. La stocker en format dense gaspille de la memoire.

In [ ]:
from scipy.sparse import csr_matrix, csc_matrix, lil_matrix

# Creation depuis un tableau dense
A = np.array([[1, 0, 0, 0], [0, 0, 3, 0], [0, 0, 0, 0], [0, 4, 0, 5]])
A_sparse = csr_matrix(A)
A_sparse

In [ ]:
A_sparse.toarray()

### Formats principaux

| Format | Usage |
|---|---|
| `csr_matrix` | Compressed Sparse Row -- operations par ligne, produit matrice-vecteur |
| `csc_matrix` | Compressed Sparse Column -- operations par colonne |
| `lil_matrix` | List of Lists -- construction incrementale |
| `coo_matrix` | Coordinate -- construction depuis des triplets (i, j, v) |

In [ ]:
from scipy.sparse import eye as speye

# Matrice identite creuse 1000x1000
I = speye(1000, format='csr')
I

### Resoudre un systeme creux

In [ ]:
from scipy.sparse.linalg import spsolve

A = csr_matrix([[3, 1, 0], [1, 4, 1], [0, 1, 3]], dtype=float)
b = np.array([1.0, 2.0, 3.0])
spsolve(A, b)

---

## 12. `scipy.spatial` -- distances et arbres

### Matrice de distances

In [ ]:
from scipy.spatial.distance import cdist

points_a = np.array([[0, 0], [1, 1]])
points_b = np.array([[2, 2], [3, 3], [4, 4]])
cdist(points_a, points_b, metric='euclidean')

### KDTree pour les recherches de voisins

In [ ]:
from scipy.spatial import KDTree

rng = np.random.default_rng(42)
points = rng.random((1000, 2))
tree = KDTree(points)

# 3 plus proches voisins du point [0.5, 0.5]
distances, indices = tree.query([0.5, 0.5], k=3)
indices

### Enveloppe convexe

In [ ]:
from scipy.spatial import ConvexHull

points = rng.random((30, 2))
hull = ConvexHull(points)
len(hull.vertices)

---

## Synthese

| Module | A retenir |
|---|---|
| `optimize` | `brentq` (racines), `minimize` (multivarie), `curve_fit` (ajustement) |
| `interpolate` | `CubicSpline` preferable a `interp1d` |
| `integrate` | `quad` (integrale definie), `solve_ivp` (EDO) |
| `linalg` | `lu`, `svd`, `cholesky`, `solve_banded` |
| `stats` | Distributions (`norm`, ...), tests (`ttest_ind`, `shapiro`, `chi2_contingency`) |
| `signal` | `butter` + `sosfilt` (filtrage), `find_peaks` |
| `fft` | `fft` + `fftfreq` pour l'analyse spectrale |
| `sparse` | `csr_matrix` pour le stockage, `spsolve` pour la resolution |
| `spatial` | `cdist` (distances), `KDTree` (voisins), `ConvexHull` |

### Regles a retenir

- **Importer le sous-module**, pas tout SciPy.
- **`solve_ivp`** remplace `odeint` (API moderne).
- **`CubicSpline`** remplace `interp1d(kind='cubic')`.
- **Matrices creuses** : choisir le bon format (`csr` pour le calcul, `lil` pour la construction).
- **Tests statistiques** : toujours verifier les hypotheses (normalite, independance) avant d'interpreter le p-value.

---

## Exercices

### Exercice 1 -- Ajustement polynomial

Generez 50 points bruites suivant `y = 2*x^2 - 3*x + 1 + bruit`. Utilisez `curve_fit` pour retrouver les coefficients `a`, `b`, `c`.

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

def poly2(x, a, b, c):
    ...

# TODO : generer les donnees bruitees et ajuster

<details>
<summary>Voir la correction</summary>

```python
import numpy as np
from scipy.optimize import curve_fit

def poly2(x, a, b, c):
    return a * x**2 + b * x + c

rng = np.random.default_rng(42)
x = np.linspace(-2, 3, 50)
y = 2*x**2 - 3*x + 1 + 0.5 * rng.normal(size=50)

params, cov = curve_fit(poly2, x, y)
print(f'a={params[0]:.3f}, b={params[1]:.3f}, c={params[2]:.3f}')
```

</details>

### Exercice 2 -- Integration numerique

Calculez l'integrale de `sin(x)/x` de 0.001 a 10 avec `quad`. Comparez avec la valeur theorique (environ 1.6583).

In [ ]:
from scipy.integrate import quad
import numpy as np

# TODO : definir f et calculer l'integrale

<details>
<summary>Voir la correction</summary>

```python
from scipy.integrate import quad
import numpy as np

def sinc(x):
    return np.sin(x) / x

result, error = quad(sinc, 0.001, 10)
print(f'Integrale = {result:.4f} (+/- {error:.2e})')
```

</details>

### Exercice 3 -- Test statistique

Deux groupes d'etudiants passent un test. Groupe A : notes normales (mu=12, sigma=2, n=25). Groupe B : notes normales (mu=13, sigma=2, n=25). Le test t indique-t-il une difference significative ? Verifiez d'abord la normalite avec Shapiro-Wilk.

In [ ]:
from scipy.stats import ttest_ind, shapiro
import numpy as np

# TODO : generer les donnees, tester la normalite, puis le t-test

<details>
<summary>Voir la correction</summary>

```python
from scipy.stats import ttest_ind, shapiro
import numpy as np

rng = np.random.default_rng(42)
groupe_a = rng.normal(12, 2, 25)
groupe_b = rng.normal(13, 2, 25)

_, p_a = shapiro(groupe_a)
_, p_b = shapiro(groupe_b)
print(f'Shapiro A : p={p_a:.4f}, B : p={p_b:.4f}')
print('Normalite OK' if p_a > 0.05 and p_b > 0.05 else 'Normalite rejetee')

stat, p = ttest_ind(groupe_a, groupe_b)
print(f't={stat:.3f}, p={p:.4f}')
print('Difference significative' if p < 0.05 else 'Pas de difference significative')
```

</details>

### Exercice 4 -- Matrice creuse

Construisez une matrice tridiagonale 1000x1000 (2 sur la diagonale, -1 sur les sous/sur-diagonales) en format CSR. Resolvez le systeme `A @ x = b` ou `b` est un vecteur de 1. Comparez le temps avec `numpy.linalg.solve` sur la version dense.

In [ ]:
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
import numpy as np

# TODO : construire la matrice, resoudre, comparer les temps

<details>
<summary>Voir la correction</summary>

```python
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
import numpy as np
import time

n = 1000
A_sparse = diags([-1, 2, -1], [-1, 0, 1], shape=(n, n), format='csr', dtype=float)
b = np.ones(n)

t0 = time.perf_counter()
x_sparse = spsolve(A_sparse, b)
t_sparse = time.perf_counter() - t0

A_dense = A_sparse.toarray()
t0 = time.perf_counter()
x_dense = np.linalg.solve(A_dense, b)
t_dense = time.perf_counter() - t0

print(f'Sparse : {t_sparse*1000:.1f} ms')
print(f'Dense  : {t_dense*1000:.1f} ms')
print(f'Ratio  : {t_dense/t_sparse:.1f}x')
```

</details>

### Exercice 5 -- EDO : chute avec frottements

Modelisez la chute d'un objet avec frottement visqueux : `m*dv/dt = m*g - k*v`. Avec m=1 kg, g=9.81, k=0.5. Condition initiale v(0)=0. Resolvez sur [0, 20] s avec `solve_ivp`. Quelle est la vitesse terminale ?

In [ ]:
from scipy.integrate import solve_ivp
import numpy as np

# TODO : definir l'EDO et la resoudre

<details>
<summary>Voir la correction</summary>

```python
from scipy.integrate import solve_ivp
import numpy as np

m, g, k = 1.0, 9.81, 0.5

def chute(t, v):
    return [g - (k/m) * v[0]]

sol = solve_ivp(chute, [0, 20], [0.0], max_step=0.1)
v_terminale = sol.y[0][-1]
v_theorique = m * g / k
print(f'v terminale numerique  = {v_terminale:.4f} m/s')
print(f'v terminale theorique  = {v_theorique:.4f} m/s')
```

</details>

---

## Ressources externes

- [Documentation officielle SciPy](https://docs.scipy.org/doc/scipy/)
- [SciPy Cookbook](https://scipy-cookbook.readthedocs.io/)
- [SciPy Lecture Notes](https://scipy-lectures.org/)
- [SciPy Tutorial](https://docs.scipy.org/doc/scipy/tutorial/)